# US Business Dynamics
### _An Exploratory Data Analysis of Business Growth and Change in the US_

### Project Overview

This project explores patters of US business activity using the Census Bureau's Business Dynamics Statistics dataset. This data provides measures of business activities such as employment, job creation, job desctruction, establishment openings, establishment closings, firm start-ups, and firm shut-downs. 
Its purpose is to perform EDA on a large, real-world dataset and show that process from load to insight. 

### Primary Question

**What patterns emerge when running data analysis on the BDS dataset provided by the US Census Bureau?**

### Analysis Guide

This project will progress through:
1. Data acquisition and inspection
2. Data quality assessment
3. Data cleaning and prep
4. Feature engineering
5. Univariate exploratory analysis
6. Multivariate exploratory analysis
7. Data visualization
8. Interpretation of key findings
9. Tableau visualization

## 1. Data Acquisition

### Data Source

**Source:** Us Census Bureau - Business Dynamics Statistics (BDS) dataset

The BDS provides annual measures of businesss activity in the US including employment, job creation, job desctruction, establishment openings, establishment closings, firm start-ups, and firm shut-downs. The dataset will focus on the State by Sector section of data to provide observations across three insightful dimensions:
- **Time:** Annual observations from 1978 through 2023
- **Geography:** US States
- **Industry:** NAICS industry sectors

Structurally, this allows business activity to be displayed and examined over time, industry, and geographic area, enriching the data with more meaning and depth. 

### Source Links (URLs)

US Census Bureau Business Dynamics Statistics:
https://www.census.gov/programs-surveys/bds.html

BDS Datasets:
https://www.census.gov/programs-surveys/bds/data.Datasets.html

BDS Codebook and Glossary:
https://www.census.gov/programs-surveys/bds/documentation.html

_Import necessary Python packages_

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

_Dataset URL_

https://www2.census.gov/programs-surveys/bds/tables/time-series/2023/bds2023_st_sec.csv

### _Loading the Dataset_

_The raw data in csv form will be loaded into a pandas dataframe_

In [2]:
df = pd.read_csv("data/bds2023_st_sec.csv")

### _Inspect the Data_

_Before running any analysis, we will check the dataset's dimensions and size..._

In [3]:
df.shape

(44574, 27)

_...and inspect its headers to identify the types of business metrics available for analysis. Note, while df.head() is useful to explore a dataset's dimensions, we do not use it here because pandas cannot display all headers for this specific dataset._

In [5]:
df.columns.tolist()

['year',
 'st',
 'sector',
 'firms',
 'estabs',
 'emp',
 'denom',
 'estabs_entry',
 'estabs_entry_rate',
 'estabs_exit',
 'estabs_exit_rate',
 'job_creation',
 'job_creation_births',
 'job_creation_continuers',
 'job_creation_rate_births',
 'job_creation_rate',
 'job_destruction',
 'job_destruction_deaths',
 'job_destruction_continuers',
 'job_destruction_rate_deaths',
 'job_destruction_rate',
 'net_job_creation',
 'net_job_creation_rate',
 'reallocation_rate',
 'firmdeath_firms',
 'firmdeath_estabs',
 'firmdeath_emp']

_We will inspect the dataset's structure, data types and non-null counts to determine how each variable is interpreted during import and to identify fields that may require more processing._

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44574 entries, 0 to 44573
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   year                         44574 non-null  int64
 1   st                           44574 non-null  int64
 2   sector                       44574 non-null  str  
 3   firms                        44574 non-null  str  
 4   estabs                       44574 non-null  str  
 5   emp                          44574 non-null  str  
 6   denom                        44574 non-null  str  
 7   estabs_entry                 44574 non-null  str  
 8   estabs_entry_rate            44574 non-null  str  
 9   estabs_exit                  44574 non-null  str  
 10  estabs_exit_rate             44574 non-null  str  
 11  job_creation                 44574 non-null  str  
 12  job_creation_births          44574 non-null  str  
 13  job_creation_continuers      44574 non-null  str  
 14  j

_Observation:_

_Most of the raw BDS data actually represents numeric counts or rates (int64), but pandas imported 25 of the 27 columns as strings (str), which may hamper our data analysis. Before we convert these strings to numbers, we will investigate further to see why these values are seen as strings._

_Assumption:_

_We can assume the column header "firms" should represent subsequent data as the numeric value of firms._ 

_Action:_

_Since we have an exorbitant amount of data to sift through, we will first see a light representation of how often a str type value appears rather than int64._

In [8]:
df["firms"].value_counts(dropna=False).head(20)

firms
39     49
73     47
74     47
163    44
44     44
78     42
103    42
D      42
125    41
60     41
90     41
105    41
131    41
69     40
89     40
104    40
48     40
85     40
93     39
161    39
Name: count, dtype: int64

_Then, we will create a temporary dataframe from the original while converting all data types to int64 (.to_numeric), force all non int64 values into NaN values (errors=coerce), as well as sift through the numeric data for anomalies to show us why this column may be imported as str type and how often those anomalies occur._

In [12]:
firms_numeric = pd.to_numeric(df["firms"], errors="coerce")

df.loc[firms_numeric.isna(), "firms"].value_counts(dropna=False)

firms
D    42
Name: count, dtype: int64

_The output of the cell above lists "D" as the culprit as to why our data is of str type and not int64. In perusing the BDS Glossary, we can see:_
> Disclosure Suppression – Disclosure suppressions are made when a cell has too few firms. Cells suppressed due to containing too few firms will appear as “D”.

_It is always a good idea to take into consideration auxiliary supporting documentation when assessing datasets for meaning._

_All values in the "firms" column designated as "D" are therefore **NOT** zeros, rather they are suppressed and cannot be inferred through data analysis. We will check to see if any other column within our dataframe that is of data type str contains indicators we can look up in our glossary._

_Anomaly scrounging:_

_We will start with an empty dictionary, we will convert all columns in our dataframe to numeric starting with the fourth column (df.columns[3:]) as we know the first three are of int64 type, identify which ones cannot convert, load those values into the empty dictionary and then we will display its contents._

In [15]:
BDS_indicators = {}

for column in df.columns[3:]:
    numeric_version = pd.to_numeric(df[column], errors="coerce")

    invalid_values = (
        df.loc[numeric_version.isna(), column]
        .value_counts()
        .to_dict()
    )

    if invalid_values:
        BDS_indicators[column] = invalid_values

BDS_indicators

{'firms': {'D': 42},
 'estabs': {'D': 42},
 'emp': {'D': 42},
 'denom': {'D': 43},
 'estabs_entry': {'D': 445},
 'estabs_entry_rate': {'D': 445, 'N': 1},
 'estabs_exit': {'D': 542},
 'estabs_exit_rate': {'D': 542, 'N': 1},
 'job_creation': {'D': 42},
 'job_creation_births': {'D': 445},
 'job_creation_continuers': {'D': 51},
 'job_creation_rate_births': {'D': 445, 'N': 1},
 'job_creation_rate': {'D': 42, 'N': 1},
 'job_destruction': {'D': 43},
 'job_destruction_deaths': {'D': 542},
 'job_destruction_continuers': {'D': 51},
 'job_destruction_rate_deaths': {'D': 542, 'N': 1},
 'job_destruction_rate': {'D': 43, 'N': 1},
 'net_job_creation': {'D': 43},
 'net_job_creation_rate': {'D': 43, 'N': 1},
 'reallocation_rate': {'D': 42, 'N': 1},
 'firmdeath_firms': {'D': 1339},
 'firmdeath_estabs': {'D': 1339},
 'firmdeath_emp': {'D': 1339}}

_In looking through the data ouput, we see that str values "D" and "N" appear frequently. From the glossary:_
> Rate Not Available – Rates that cannot be calculated due to a denominator of ‘0’ will appear as “N”.

_These indicators are also not evenly distributed through the dataset, with some columns containing more or less than others of these indicators. We, again, will **NOT** consider these values as zeros._

_Our dataset is ready to clean._